# Credit Card Default Risk Analytics

End-to-end **data analytics** notebook — Python cleaning, SQL analysis, a rule-based risk scorecard, and a BI export. No ML model is trained anywhere in this notebook.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as C
from src.data import build_clean
from src.sql_queries import load_to_sql, run_all_queries, run_query
from src.risk_scorecard import compute_risk_score, validate_against_actuals

sns.set_theme(style="whitegrid")
%matplotlib inline


## 1. Load and clean
Download `UCI_Credit_Card.csv` from the UCI repository (or Kaggle mirror) into `data/raw/` first.

In [ ]:
df = build_clean(force=True)
print(f"{len(df):,} clients, overall default rate {df[C.TARGET].mean():.2%}")
df.head()


## 2. EDA

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=df, x="age_group", y=C.TARGET, ax=ax, errorbar=None)
ax.set_title("Default rate by age group"); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=df, x=C.TARGET, y="utilization", ax=ax)
ax.set_xticklabels(["No default", "Default"])
ax.set_title("Utilisation vs. default outcome"); plt.show()


In [ ]:
num_cols = ["credit_limit","age","utilization","months_delinquent","worst_delinquency","payment_ratio", C.TARGET]
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.tight_layout(); plt.show()


For the full 6-figure export used in the README, run `python -m src.eda` from the terminal.

## 3. Load into SQL and run the analytical queries
This loads the cleaned table into a local SQLite database (`data/processed/credit_card.db`) and runs every named query in `sql/queries.sql`.

In [ ]:
load_to_sql(df)
results = run_all_queries()
list(results.keys())


In [ ]:
results["default_rate_by_credit_limit_decile"]


In [ ]:
results["utilization_band_summary"]


In [ ]:
results["default_rate_by_delinquency_count"]


## 4. Build the rule-based risk scorecard
Four weighted factors combine into a 0-100 score and four bands. No `.fit()` anywhere — thresholds come from the SQL output above, set by hand.

In [ ]:
scored = compute_risk_score(df)
scored[["utilization","months_delinquent","worst_delinquency","payment_ratio","risk_score","risk_band"]].head(10)


In [ ]:
scored["risk_band"].value_counts().sort_index()


## 5. Validate against actual outcomes
The ONLY place the real `default_next_month` column is used. A good scorecard shows a strictly increasing default rate from Low to Very High.

In [ ]:
validation = validate_against_actuals(scored)
validation


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=validation, x="risk_band", y="actual_default_rate_pct", ax=ax)
ax.set_title("Actual default rate by risk band")
plt.show()

monotonic = validation["actual_default_rate_pct"].is_monotonic_increasing
print(f"Monotonically increasing: {monotonic}")


## 6. Export for Tableau / Power BI

In [ ]:
scored.to_csv(C.TABLEAU_FILE, index=False)
print(f"Saved -> {C.TABLEAU_FILE}")


## 7. Build the dashboard (outside this notebook)

1. Open Tableau Public Desktop or Power BI -> connect to `tableau_export.csv`
2. **Sheet 1:** risk band distribution (bar, count per band)
3. **Sheet 2:** actual default rate by band (validates the scorecard visually)
4. **Sheet 3:** utilisation x delinquency heatmap
5. **Sheet 4:** demographic breakdown, filterable by risk band
6. Combine into a dashboard, publish, link it in `README.md`
